In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../'))

In [5]:
import pandas as pd
import warnings

import json
from numpy import random
from dataclasses import dataclass

from model.utils import report_results
from model.train import train_classifier

from sklearn.neural_network import MLPClassifier

DEFAULT_RANDOM_SEED = 774
random.mtrand._rand.seed(DEFAULT_RANDOM_SEED)
seed_list = random.random_integers(low=0, high=2**32 - 1, size=5)
warnings.filterwarnings("ignore")

search_params = {"learning_rate": (0.05, 0.1, 0.5, 1), "max_features": (0.1, 0.2, 0.5, "sqrt"), "l2_regularization": (0, 0.5, 1), "max_depth": (16, 32, 64, None), "class_weight": ("balanced",)}

/var/folders/vp/tqkr07mj3cn274npl2jmvy5w0000gq/T/ipykernel_93315/709193056.py:15: DeprecationWarning: This function is deprecated. Please call randint(0, 4294967295 + 1) instead
  seed_list = random.random_integers(low=0, high=2**32 - 1, size=5)


In [6]:
@dataclass
class RunConfiguration:
  run_grid_search: bool
  default_parameters: dict

In [7]:
def get_parameters(df: pd.DataFrame, run_config: RunConfiguration):
  if not run_config.run_grid_search:
    return run_config.default_parameters
  
  grid_search_response = train_classifier(MLPClassifier, target="subtype", data=df, grid_search_params=search_params)
  parameters = {k: grid_search_response.model.get_params()[k] for k in grid_search_response.model.get_params().keys() & search_params.keys() }
  print(parameters)
  return parameters

def run_tests(category: str, method: str, metric: str, run_config: RunConfiguration):
  data = pd.read_csv(f"../../preprocessed/{category}/genes.csv").drop(columns=["sample_id"])
  pvalues = json.loads(open(f"../../preprocessed/{category}/important_genes_{method}_{metric}.json").readline())

  chosen_genes = list(set([y["gene"] for x in [sex_values[:15] for subtype_items in pvalues.values() for sex_values in subtype_items.values()] for y in x]))
  print(f"Total chosen genes: {len(chosen_genes)}")

  df = data[["subtype", "sex", *chosen_genes]]
  results = report_results(df, MLPClassifier, get_parameters(df, run_config), seed_list)
  print(results.report)

In [40]:
run_tests(
  category="min_tpm_5",
  method="random_forest",
  metric="recall",
  run_config=RunConfiguration(
    run_grid_search=False,
    default_parameters=dict(hidden_layer_sizes=(32, 32), learning_rate_init=0.0001, alpha=2.5, max_iter=1000)
  )
)

Total chosen genes: 242


 20%|██        | 1/5 [00:08<00:33,  8.50s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.86      0.93        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.90      0.97      0.93        58
        HYPO       0.89      0.86      0.87        28
       KMT2A       1.00      0.98      0.99        46
        PAX5       0.92      1.00      0.96        33
      PHlike       0.90      0.95      0.92        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.73      0.85        15

    accuracy                           0.96       427
   macro avg       0.96      0.93      0.94       427
weighted avg       0.96      0.96      0.96       427



 40%|████      | 2/5 [00:17<00:25,  8.65s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.83      0.91        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       0.99      1.00      0.99        81
       HYPER       0.93      0.97      0.95        58
        HYPO       0.89      0.86      0.87        28
       KMT2A       1.00      0.98      0.99        46
        PAX5       0.86      0.97      0.91        33
      PHlike       0.90      0.93      0.91        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.87      0.93        15

    accuracy                           0.96       427
   macro avg       0.96      0.94      0.95       427
weighted avg       0.96      0.96      0.96       427



...

              precision    recall  f1-score   support

     BCRABL1       1.00      0.86      0.93        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       0.98      1.00      0.99        81
       HYPER       0.92      0.93      0.92        58
        HYPO       0.86      0.89      0.88        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       0.91      0.94      0.93        33
      PHlike       0.88      0.93      0.91        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.73      0.85        15

    accuracy                           0.95       427
   macro avg       0.95      0.93      0.94       427
weighted avg       0.95      0.95      0.95       427



 80%|████████  | 4/5 [00:31<00:07,  7.59s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.86      0.93        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       0.96      1.00      0.98        81
       HYPER       0.92      0.95      0.93        58
        HYPO       0.93      0.89      0.91        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       0.92      1.00      0.96        33
      PHlike       0.96      0.93      0.95        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       0.93      0.87      0.90        15

    accuracy                           0.96       427
   macro avg       0.96      0.95      0.95       427
weighted avg       0.96      0.96      0.96       427



100%|██████████| 5/5 [00:36<00:00,  7.36s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.83      0.91        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.90      0.97      0.93        58
        HYPO       0.90      0.93      0.92        29
       KMT2A       1.00      1.00      1.00        46
        PAX5       0.91      0.97      0.94        32
      PHlike       0.90      0.93      0.91        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.73      0.85        15

    accuracy                           0.96       427
   macro avg       0.96      0.94      0.95       427
weighted avg       0.96      0.96      0.96       427

           Metric          Overall             Male           Female
0   F1 (Weighted)  0.8718 ± 0.0203  0.8803 ± 0.0246  0.8448 ± 0.0289
1      F1 (Macro)  0.8405 ± 0.0269  0.8675 ± 0.0270  0.8009 ± 0.0424
2  Recall (Macro)  0.8300 ± 0.0284

In [42]:
run_tests(
  category="min_tpm_5",
  method="random_forest",
  metric="f1",
  run_config=RunConfiguration(
    run_grid_search=False,
    default_parameters=dict(hidden_layer_sizes=(32, 32), learning_rate_init=0.0001, alpha=2.5, max_iter=1000)
  )
)

Total chosen genes: 242


 20%|██        | 1/5 [00:05<00:22,  5.59s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.86      0.93        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.90      0.97      0.93        58
        HYPO       0.89      0.86      0.87        28
       KMT2A       1.00      0.98      0.99        46
        PAX5       0.92      1.00      0.96        33
      PHlike       0.90      0.95      0.92        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.73      0.85        15

    accuracy                           0.96       427
   macro avg       0.96      0.93      0.94       427
weighted avg       0.96      0.96      0.96       427



 40%|████      | 2/5 [00:10<00:14,  4.99s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.83      0.91        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       0.99      1.00      0.99        81
       HYPER       0.93      0.97      0.95        58
        HYPO       0.89      0.86      0.87        28
       KMT2A       1.00      0.98      0.99        46
        PAX5       0.86      0.97      0.91        33
      PHlike       0.90      0.93      0.91        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.87      0.93        15

    accuracy                           0.96       427
   macro avg       0.96      0.94      0.95       427
weighted avg       0.96      0.96      0.96       427



...

              precision    recall  f1-score   support

     BCRABL1       1.00      0.86      0.93        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       0.98      1.00      0.99        81
       HYPER       0.92      0.93      0.92        58
        HYPO       0.86      0.89      0.88        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       0.91      0.94      0.93        33
      PHlike       0.88      0.93      0.91        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.73      0.85        15

    accuracy                           0.95       427
   macro avg       0.95      0.93      0.94       427
weighted avg       0.95      0.95      0.95       427



 80%|████████  | 4/5 [00:20<00:05,  5.28s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.86      0.93        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       0.96      1.00      0.98        81
       HYPER       0.92      0.95      0.93        58
        HYPO       0.93      0.89      0.91        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       0.92      1.00      0.96        33
      PHlike       0.96      0.93      0.95        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       0.93      0.87      0.90        15

    accuracy                           0.96       427
   macro avg       0.96      0.95      0.95       427
weighted avg       0.96      0.96      0.96       427



100%|██████████| 5/5 [00:27<00:00,  5.42s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.83      0.91        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.90      0.97      0.93        58
        HYPO       0.90      0.93      0.92        29
       KMT2A       1.00      1.00      1.00        46
        PAX5       0.91      0.97      0.94        32
      PHlike       0.90      0.93      0.91        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.73      0.85        15

    accuracy                           0.96       427
   macro avg       0.96      0.94      0.95       427
weighted avg       0.96      0.96      0.96       427

           Metric          Overall             Male           Female
0   F1 (Weighted)  0.8718 ± 0.0203  0.8803 ± 0.0246  0.8448 ± 0.0289
1      F1 (Macro)  0.8405 ± 0.0269  0.8675 ± 0.0270  0.8009 ± 0.0424
2  Recall (Macro)  0.8300 ± 0.0284

In [43]:
run_tests(
  category="min_tpm_5",
  method="logistic",
  metric="recall",
  run_config=RunConfiguration(
    run_grid_search=False,
    default_parameters=dict(hidden_layer_sizes=(32, 32), learning_rate_init=0.0001, alpha=2.5, max_iter=1000)
  )
)

Total chosen genes: 252


 20%|██        | 1/5 [00:05<00:20,  5.22s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.97      0.98        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.98      1.00      0.99        58
        HYPO       0.97      1.00      0.98        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       1.00      1.00      1.00        33
      PHlike       1.00      1.00      1.00        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.93      0.97        15

    accuracy                           1.00       427
   macro avg       0.99      0.99      0.99       427
weighted avg       1.00      1.00      1.00       427



 40%|████      | 2/5 [00:09<00:13,  4.54s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      1.00      1.00        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       1.00      1.00      1.00        58
        HYPO       1.00      1.00      1.00        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       1.00      1.00      1.00        33
      PHlike       0.98      1.00      0.99        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.93      0.97        15

    accuracy                           1.00       427
   macro avg       1.00      0.99      1.00       427
weighted avg       1.00      1.00      1.00       427



...

              precision    recall  f1-score   support

     BCRABL1       1.00      1.00      1.00        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.98      1.00      0.99        58
        HYPO       1.00      1.00      1.00        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       1.00      1.00      1.00        33
      PHlike       1.00      1.00      1.00        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.93      0.97        15

    accuracy                           1.00       427
   macro avg       1.00      0.99      1.00       427
weighted avg       1.00      1.00      1.00       427



 80%|████████  | 4/5 [00:19<00:04,  4.81s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      1.00      1.00        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.98      1.00      0.99        58
        HYPO       1.00      1.00      1.00        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       1.00      1.00      1.00        33
      PHlike       1.00      1.00      1.00        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.93      0.97        15

    accuracy                           1.00       427
   macro avg       1.00      0.99      1.00       427
weighted avg       1.00      1.00      1.00       427



100%|██████████| 5/5 [00:23<00:00,  4.69s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      1.00      1.00        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.98      1.00      0.99        58
        HYPO       1.00      1.00      1.00        29
       KMT2A       1.00      1.00      1.00        46
        PAX5       1.00      1.00      1.00        32
      PHlike       1.00      1.00      1.00        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.93      0.97        15

    accuracy                           1.00       427
   macro avg       1.00      0.99      1.00       427
weighted avg       1.00      1.00      1.00       427

           Metric          Overall             Male           Female
0   F1 (Weighted)  0.9082 ± 0.0134  0.9205 ± 0.0095  0.8943 ± 0.0206
1      F1 (Macro)  0.8949 ± 0.0254  0.9075 ± 0.0353  0.8605 ± 0.0271
2  Recall (Macro)  0.8863 ± 0.0260

In [37]:
run_tests(
  category="min_tpm_5",
  method="logistic",
  metric="f1",
  run_config=RunConfiguration(
    run_grid_search=False,
    default_parameters=dict(hidden_layer_sizes=(32, 32), learning_rate_init=0.0001, alpha=2.5, max_iter=1000)
  )
)

Total chosen genes: 252


 20%|██        | 1/5 [00:10<00:40, 10.13s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.97      0.98        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       1.00      1.00      1.00        58
        HYPO       0.97      1.00      0.98        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       1.00      1.00      1.00        33
      PHlike       1.00      1.00      1.00        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      1.00      1.00        15

    accuracy                           1.00       427
   macro avg       1.00      1.00      1.00       427
weighted avg       1.00      1.00      1.00       427



 40%|████      | 2/5 [00:20<00:30, 10.02s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.97      0.98        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       1.00      1.00      1.00        58
        HYPO       1.00      1.00      1.00        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       1.00      1.00      1.00        33
      PHlike       0.97      1.00      0.98        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.93      0.97        15

    accuracy                           1.00       427
   macro avg       1.00      0.99      0.99       427
weighted avg       1.00      1.00      1.00       427



...

              precision    recall  f1-score   support

     BCRABL1       1.00      1.00      1.00        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.98      1.00      0.99        58
        HYPO       1.00      1.00      1.00        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       1.00      1.00      1.00        33
      PHlike       1.00      1.00      1.00        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.93      0.97        15

    accuracy                           1.00       427
   macro avg       1.00      0.99      1.00       427
weighted avg       1.00      1.00      1.00       427



 80%|████████  | 4/5 [00:37<00:09,  9.11s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      1.00      1.00        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.98      1.00      0.99        58
        HYPO       1.00      1.00      1.00        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       1.00      1.00      1.00        33
      PHlike       1.00      1.00      1.00        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.93      0.97        15

    accuracy                           1.00       427
   macro avg       1.00      0.99      1.00       427
weighted avg       1.00      1.00      1.00       427



100%|██████████| 5/5 [00:45<00:00,  9.16s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      1.00      1.00        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.98      1.00      0.99        58
        HYPO       1.00      1.00      1.00        29
       KMT2A       1.00      1.00      1.00        46
        PAX5       1.00      1.00      1.00        32
      PHlike       1.00      1.00      1.00        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.93      0.97        15

    accuracy                           1.00       427
   macro avg       1.00      0.99      1.00       427
weighted avg       1.00      1.00      1.00       427

           Metric          Overall             Male           Female
0   F1 (Weighted)  0.9157 ± 0.0117  0.9076 ± 0.0180  0.9121 ± 0.0159
1      F1 (Macro)  0.9006 ± 0.0235  0.8954 ± 0.0440  0.8824 ± 0.0190
2  Recall (Macro)  0.8863 ± 0.0260

In [46]:
run_tests(
  category="min_tpm_5",
  method="logistic",
  metric="f1",
  run_config=RunConfiguration(
    run_grid_search=False,
    default_parameters=dict(hidden_layer_sizes=(32, 32, 32, 32), learning_rate_init=0.0001, alpha=5, max_iter=1000)
  )
)

Total chosen genes: 252


 20%|██        | 1/5 [00:05<00:23,  5.86s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.90      0.95        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.92      1.00      0.96        58
        HYPO       0.70      0.93      0.80        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       0.97      1.00      0.99        33
      PHlike       0.95      1.00      0.97        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       0.00      0.00      0.00        15

    accuracy                           0.95       427
   macro avg       0.85      0.88      0.87       427
weighted avg       0.93      0.95      0.94       427



 40%|████      | 2/5 [00:11<00:16,  5.63s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.86      0.93        29
     DUX4IGH       0.98      1.00      0.99        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.98      1.00      0.99        58
        HYPO       0.97      1.00      0.98        28
       KMT2A       0.98      1.00      0.99        46
        PAX5       0.97      1.00      0.99        33
      PHlike       0.95      0.98      0.97        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       1.00      0.80      0.89        15

    accuracy                           0.98       427
   macro avg       0.98      0.96      0.97       427
weighted avg       0.98      0.98      0.98       427



...

              precision    recall  f1-score   support

     BCRABL1       1.00      0.86      0.93        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.83      1.00      0.91        58
        HYPO       0.92      0.86      0.89        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       0.97      1.00      0.99        33
      PHlike       0.88      1.00      0.93        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       0.00      0.00      0.00        15

    accuracy                           0.95       427
   macro avg       0.86      0.87      0.86       427
weighted avg       0.92      0.95      0.93       427



 80%|████████  | 4/5 [00:22<00:05,  5.57s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.90      0.95        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.84      1.00      0.91        58
        HYPO       0.96      0.93      0.95        28
       KMT2A       1.00      1.00      1.00        46
        PAX5       0.97      1.00      0.99        33
      PHlike       0.89      1.00      0.94        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       0.00      0.00      0.00        15

    accuracy                           0.95       427
   macro avg       0.87      0.88      0.87       427
weighted avg       0.92      0.95      0.94       427



100%|██████████| 5/5 [00:28<00:00,  5.66s/it]

              precision    recall  f1-score   support

     BCRABL1       1.00      0.86      0.93        29
     DUX4IGH       1.00      1.00      1.00        52
   ETV6RUNX1       1.00      1.00      1.00        81
       HYPER       0.88      1.00      0.94        58
        HYPO       1.00      1.00      1.00        29
       KMT2A       1.00      1.00      1.00        46
        PAX5       1.00      1.00      1.00        32
      PHlike       0.84      1.00      0.91        57
    TCF3PBX1       1.00      1.00      1.00        28
      iAMP21       0.00      0.00      0.00        15

    accuracy                           0.96       427
   macro avg       0.87      0.89      0.88       427
weighted avg       0.93      0.96      0.94       427

           Metric          Overall             Male           Female
0   F1 (Weighted)  0.8739 ± 0.0186  0.8819 ± 0.0256  0.8614 ± 0.0135
1      F1 (Macro)  0.8119 ± 0.0328  0.8038 ± 0.0411  0.8010 ± 0.0291
2  Recall (Macro)  0.8012 ± 0.0269